# Initialization

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, trim, regexp_replace

# Reading Bronze Table

In [0]:
df = spark.table("workspace.bronze.erp_loc_a101")


# Data Transformations

## Trimming

In [0]:
for field in df.schema.fields:
  if isinstance(field.dataType, StringType):
      if field.name == "CID":
          df = df.withColumn(field.name, regexp_replace(col(field.name), "-", ""))
      else:
        df = df.withColumn(field.name, trim(col(field.name)))

## Normalization

In [0]:
df = (
    df
    .withColumn("CNTRY",
    F.when(F.upper(F.col("CNTRY")).isin("US", "USA", "United States"), "United States")
    .when(F.upper(F.col("CNTRY")).isin("DE", "Germany"), "Germany")
    .when(F.upper(F.col("CNTRY")).isin("", " ", "  "), "n/a")
    .when(F.col("CNTRY").isNull(), "n/a")
    .otherwise(F.col("CNTRY")))
)

df.groupBy("CNTRY").count().show()

## Renaming Columns

In [0]:
RENAME_MAP = {
    "CID": "customer_id",
    "CNTRY": "country"}

for old_name, new_name in RENAME_MAP.items():
    df = df.withColumnRenamed(old_name, new_name)

# Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("silver.erp_customer_country")

In [0]:
%sql
SELECT * FROM workspace.silver.erp_customer_country LIMIT 15